# Imports, Defaults and Initializations
Clicking on the below cell is mandatory. Click on the cell below and press `Shift + Enter`. To run any cell, use the same keyboard shortcut.

In [ ]:
import backtrader
from helpers.datahandler import *
from helpers.investing import *
# from helpers.journal_handler import *
from helpers.intraday import *
from helpers.nse_data import *
from helpers.backtest import *
from helpers.FnO import analyse_option_chain, get_next_expiry_date
# from helpers.online_brokers import KiteZerodha # Yes! Finally we can have live Data for Free!!!


pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 25)


In [ ]:
# 1. Initialize the DataHandler
DH = DataHandler()

# 2. Get the list of all registered stocks and start the download
stocks_to_download = DH.data['registered_stocks']
# stocks_to_download = DH.all_stocks
print(f"Starting download for {len(stocks_to_download)} stocks. This may take a while...")
DH.multiprocess_download_stocks(stocks_to_download)
DH.update_fresh_files()
print("Download complete!")


### Prune Unregistered Stocks 

In [ ]:
DH = DataHandler()
# data_path = './data', check_fresh = True)
# DH.all_stocks
# DH.multiprocess_download_stocks(DH.all_stocks)
# DH.download_new('OLAELEC')
# DH.update_stock_data('OLAELEC')
# DH.update_fresh_files()
# DH.update_new_listings()
DH.prune_unregistered_stocks()


### Bhav Copy download

In [ ]:
from helpers.datahandler import DataHandler
from datetime import date

# Initialize the handler
dh = DataHandler()

# 1. Download Bhavcopy for the last 7 days (default)
dh.download_bhavcopy(days=2)

# # 2. Download for a specific date range # start = date(2025, 11, 21) # end = date(2025, 11, 22)
# dh.download_bhavcopy(start_date=start, end_date=end)

# # 3. Download for the last 30 days from today
# dh.download_bhavcopy(days=30)

# 4. Mearge newly downloaded data with existing stock files (DRY RUN: preview only)
# NOTE: default is a dry-run so you don't accidentally overwrite data
# dh.update_stocks_with_bhavcopy_data(days=3, dry_run=True)
# Use compute_52w=True to compute and update 52-week high/low in the per-stock files
# dh.update_stocks_with_bhavcopy_data(days=1, dry_run=True, compute_52w=True)
# To actually persist changes (write CSV files and update mapping):
dh.update_stocks_with_bhavcopy_data(days=1, dry_run=False, compute_52w=True)



### Intialzing all modules...

In [ ]:
# 1. Initialize the DataHandler
DH = DataHandler()

In = Investing(check_fresh = False) # Investing or Swing trading analysis
Intra = IntraDay() # Intraday strategies and analysis
NSE = NSEData() # Get NSE Live data
MS = MarketSentiment()
BT = Backtest()
ISS  = IntradayStockSelection() # Few of the Intraday Stock Selection methods. Not methods, but just a few things like stats of Move, Range etc

In.update_new_listings() # Good to run once in a while say after every 1 month
In.update_fresh_nifty_indices()
In.update_FnO() # Future and Options List

In [ ]:
# Demo: check CA bundle and SSL fallbacks
# This cell demonstrates how to use MarketSentiment.check_ca(), certifi, and the trust_all_ssl fallback.
import certifi
from helpers.nse_data import MarketSentiment

ms = MarketSentiment()
print('check_ca()', ms.check_ca())

# Use certifi's CA bundle explicitly (recommended if your OS lacks CA certs)
ms_cert = MarketSentiment(ca_bundle_path=certifi.where())
print('Using certifi bundle -> get_live_sentiment():', ms_cert.get_live_sentiment())

# Explicit insecure fallback (not recommended)
ms_insecure = MarketSentiment(trust_all_ssl=True)
print('With trust_all_ssl -> get_live_sentiment():', ms_insecure.get_live_sentiment())

In [ ]:
# %pip install google-auth google-api-python-client

# [Journal](https://drive.google.com/file/d/1JipUU6Im1YVKSdufw4VHitwS010nFigL/view)
Make a journal in Google Drive for each trade you take to analyse. **It is a not so simple process and need to connect with Google API so if you're not comfortable with it, just skip it**

In [ ]:
private_key = join(expanduser('~'),'Documents','client_secret.json') # Personal for each user. Mine is stored in /home/Documents
JH = JournalHandler(private_key)

journal = JH.get_journal('Finance Journal') # My Google Sheet jornam has name "Finance Journal"

JH.total_pl(journal) # total Profit and Loss up until now

# [Swing  Trading Strategies](https://www.investopedia.com/terms/s/swingtrading.asp)
## Tweak parameters by reading the Docstring
## Basic Details
Basic Details about stocks such as Momentum, Ichimoku, 1-2-3 candles pattern etc

In [ ]:
budget = 200000

df = In.get_recent_info(nifty=500, **{'mvs':[20,50,100,200]})
# df = In.get_recent_info(nifty = 500, custom_list= ['GODIGIT', 'M&M', 'NAVINFLUOR', 'ALKEM', 'LT', 'PNB', 'CUMMINSIND', 'CAMS', 'SHRIRAMFIN', 'HDFCLIFE', 'ABCAPITAL', 'CANFINHOME', 'REDINGTON', 'RADICO', 'SBIN', 'AIAENG', 'BAJAJ-AUTO', 'NAM-INDIA', 'BHARTIARTL', 'MARUTI', 'BANKINDIA', 'EMCURE', 'ONGC', 'POWERINDIA', 'AUBANK', 'VIJAYA'], **{'mvs':[20,50,100,200]})

df.head(20) # show first 4 values



## Breakout
Stocks that might breakout. Includes only those stocks which are above 50 days SMA, have atleast `n` number of touches within the `p%` of the recent candle. Just look at these stocks manually. Change the parameters to control the number of stocks to scan.

In [ ]:
# In.tight_consolidation_stocks(stocks = 'nifty_500', diff = 0.007, min_count = 5, lookback_period = 10) 
In.tight_consolidation_stocks(stocks = 'nifty_500', diff = 0.008, min_count = 5, lookback_period = 10) 
# There should be atleast 5 candles touching the +- 0.75% of the line within 7 days

## 44-SMA 
If you find a `Green` candle taking support on 44 days `Simple Moving Average` line and stock is in uptrend, you can take the trade and set Risk to Reward ratio as 1:2

In [ ]:
budget = 200000 # Total Budget
risk = 400 # Risk per trade
df = In.show_full_stats(budget = budget,risk=risk, diff = 150, nifty='nifty_500') # Show only stocks undder your budget and  whose recent candle's lowest is not more than 151 rupees above the moving average line
df[(df['RSI Value'] < 80) & (df['CCI Value'] < 250)]

## Golden Crossover
When 44 days `SMA` Crosses 200 days `SMA` from below and you find a `Green` candle within 15 days taking support on `44` days line set Risk to Reward ratio as 1:5

In [ ]:
for name in In.data['nifty_500']:
    df = In.open_downloaded_stock(name)
    if In.has_golden_crossover(df,lookback=21):
        print(name)

## Golden Cross & Death Cross Analysis (50/200 SMA)
This section uses the modified `DataHandler` to find recent Golden Crosses (50-day SMA crosses above 200-day SMA) and Death Crosses (50-day SMA crosses below 200-day SMA) using your local CSV data.

In [ ]:
from helpers.gdcross_datahandler import DataHandler

# You can use the full nifty_500 list, but a smaller list is faster for testing.
stock_list_to_scan = In.data['nifty_500'] 
# stock_list_to_scan = ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'ICICIBANK', 'SBIN', 'BAJFINANCE', 'MARUTI', 'ONGC', 'ITC']

# Initialize the handler with 50-day and 200-day SMAs for long-term crossover analysis
crossover_handler = DataHandler(stock_list=stock_list_to_scan, short_sma=50, long_sma=200)

# --- 1. Find Golden Crosses (Bullish) ---
print("--- Searching for recent Golden Crosses ---")
golden_crosses_df = crossover_handler.find_recent_golden_crossovers(days_back=30)
display(golden_crosses_df)

# --- 2. Find Death Crosses (Bearish) ---
print("\n--- Searching for recent Death Crosses ---")
death_crosses_df = crossover_handler.find_recent_death_crossovers(days_back=30)
display(death_crosses_df)

### Plot Top Crossover Results

In [ ]:
# Plot the chart for the top Golden Cross result
if isinstance(golden_crosses_df, pd.DataFrame) and not golden_crosses_df.empty:
    top_golden_cross_stock = golden_crosses_df.iloc[0]['Stock']
    print(f"\n--- Plotting Top Golden Cross: {top_golden_cross_stock} ---")
    crossover_handler.plot_stock_chart(top_golden_cross_stock, days_back=30)
else:
    print("\nNo Golden Crosses to plot.")

# Plot the chart for the top Death Cross result
if isinstance(death_crosses_df, pd.DataFrame) and not death_crosses_df.empty:
    top_death_cross_stock = death_crosses_df.iloc[0]['Stock']
    print(f"\n--- Plotting Top Death Cross: {top_death_cross_stock} ---")
    crossover_handler.plot_stock_chart_death_cross(top_death_cross_stock, days_back=30)
else:
    print("\nNo Death Crosses to plot.")

## RSI Signal: Oversold / Overbought Stocks
Buy when RSI is less than 30 sell when greater than 70 ("Buy"/"Sell")

In [ ]:
for name in In.data['nifty_500']:
    df = In.open_downloaded_stock(name)
    if In.get_RSI(df,signal_only=True) == "Buy":
        print(name)

## MACD Signal
It is the strategy when MACD (Blue) Line cuts the Signal (Red) line from below then it is a buy signal. When the blue line cuts from upside down, it is sell signal

In [ ]:
for name in In.data['nifty_500']:
    df = In.open_downloaded_stock(name)
    if In.macd_signal(df) == 'Buy':
        print(name)

## CCI Signal : Rally
**Faster than MACD Signal works closely with Stochastic RSI**

When CCI values comes from below  -100 to above -100, buy signal and when it comes from above +100 to below +100, sell signal

In [ ]:
for name in In.data['nifty_500']:
    df = In.open_downloaded_stock(name)
    if In.get_CCI(df, signal_only = True) == 'Buy':
        print((name, In.get_index(name)))

# Stochastic Oscillator
1. Buy when the fast line cuts the slow line from below in an OVERSOLD zone ( both are below 20). Wait for both lines to go above Oversold and then buy. Recent candle closing must be above 200-MA

2. Sell when both lines reaches Overbought region (above 70) and fast line crosses slow from above. Wait for both lines to go below the threshold

In [ ]:
for name in In.data['nifty_500']:
    if In.Stochastic(In.open_downloaded_stock(name), signal_only = True) == "Buy":
        print(name, In.get_index(name))

# Intraday 
## See documentations for Intraday strategies

## Pivots Points
As discusses in the book, [How to Make Money in Intraday Trading by Ashwani Gujral](https://www.amazon.in/How-Make-Money-Intraday-Trading/dp/9386268159) (go to `./ebooks` directory to find the annoted version of this book), this one is very important indicator. None of the online screening platforms have support for displaying [`CPR`](https://zerodha.com/varsity/chapter/the-central-pivot-range/) by default and most of these do not even show the Pivots, Support, Resistance until `9:15`. This one gives you the values of `Pivot with CPR, 3 Supports and Resistances` at the day end as long as the NSE updates the day end data so that you can plan your day in advance.

In [ ]:
data = In.open_live_stock_data('OKPLA')
pivot_data = In.get_Pivot_Points(data, plot = True, num_days_back=5) # access pivot_data to see exact values of support resistances and pivots and pivots boundries

## Narrow Range (Stock Selection)
If recent candle's range is lowest among `X` other days then the future candle might break it's low/high

In [ ]:
for name in In.data['nifty_500']:
    if Intra.NR_strategy(name, range_ = 20):
        print(Intra.prob_by_percent_change(symbol = [name], index = None, time_period = 10, change_percent=0.07),In.get_index(name,'all'))

## Whole Number Open (Stock Selection + Strategy)
When Open == Low / High in Whole number as `xxx.00` within 15 minutes

In [ ]:
Intra.whole_number_strategy(nifty=500, min_val=101, max_val=5000, print_results = True)

In [ ]:
# Store results for visualization
results = Intra.whole_number_strategy(nifty=500, min_val=101, max_val=5000, print_results=False)

# Helper function to convert results to DataFrame
def format_whole_number_results(results_dict):
    """Convert whole_number_strategy results to DataFrame for visualization"""
    all_rows = []
    
    for signal_type, stocks_list in results_dict.items():
        for stock_tuple in stocks_list:
            all_rows.append({
                'Signal': signal_type,
                'Stock': stock_tuple[0],
                'Change': round(float(stock_tuple[1]), 2),
                'ATR': round(float(stock_tuple[2]), 2) if not pd.isna(stock_tuple[2]) else None,
                'Change%': float(stock_tuple[3].rstrip('%')),
                'Remaining Move': round(float(stock_tuple[4]), 2) if not pd.isna(stock_tuple[4]) else None,
                'Index': stock_tuple[5]
            })
    
    df = pd.DataFrame(all_rows)
    return df

df_results = format_whole_number_results(results)
print(f"Total opportunities found: {len(df_results)} ({len(results['Long'])} Long, {len(results['Short'])} Short)")


## Formatted Results Table
Below is a clean table view of all opportunities sorted by signal type and remaining move potential


In [ ]:
# Option 1: Detailed table sorted by Remaining Move (best opportunity first)
df_sorted = df_results.sort_values('Remaining Move', ascending=False, na_position='last')
print("=" * 120)
print("OPTION 1: ALL OPPORTUNITIES (sorted by Remaining Move - Higher is Better)")
print("=" * 120)
display(df_sorted.style.format({
    'Change': '{:.2f}',
    'ATR': '{:.2f}',
    'Change%': '{:.2f}%',
    'Remaining Move': '{:.2f}'
}).background_gradient(subset=['Change%', 'Remaining Move'], cmap='RdYlGn').hide(axis='index'))

In [ ]:
# Option 2: Separate tables for Long and Short signals
print("\n" + "=" * 120)
print("OPTION 2A: LONG SIGNALS (Buy Opportunities - sorted by Remaining Move)")
print("=" * 120)
df_long = df_results[df_results['Signal'] == 'Long'].sort_values('Remaining Move', ascending=False, na_position='last')
display(df_long[['Stock', 'Change', 'ATR', 'Change%', 'Remaining Move', 'Index']].style.format({
    'Change': '{:.2f}',
    'ATR': '{:.2f}',
    'Change%': '{:.2f}%',
    'Remaining Move': '{:.2f}'
}).background_gradient(subset=['Remaining Move'], cmap='Greens').hide(axis='index'))

print("\n" + "=" * 120)
print("OPTION 2B: SHORT SIGNALS (Sell Opportunities - sorted by Remaining Move)")
print("=" * 120)
df_short = df_results[df_results['Signal'] == 'Short'].sort_values('Remaining Move', ascending=False, na_position='last')
display(df_short[['Stock', 'Change', 'ATR', 'Change%', 'Remaining Move', 'Index']].style.format({
    'Change': '{:.2f}',
    'ATR': '{:.2f}',
    'Change%': '{:.2f}%',
    'Remaining Move': '{:.2f}'
}).background_gradient(subset=['Remaining Move'], cmap='Reds').hide(axis='index'))

## Interactive Charts & Visualizations


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chart 1: Remaining Move Potential - Top stocks by opportunity
fig1 = px.bar(
    df_results.dropna(subset=['Remaining Move']).sort_values('Remaining Move', ascending=True).tail(15),
    x='Remaining Move', 
    y='Stock', 
    color='Signal',
    orientation='h',
    color_discrete_map={'Long': '#00cc96', 'Short': '#ef553b'},
    title='Top 15 Stocks by Remaining Move Potential (Best Opportunities)',
    labels={'Remaining Move': 'Remaining Move Score', 'Stock': 'Stock Symbol'},
    height=600
)
fig1.update_layout(showlegend=True, hovermode='closest')
fig1.show()


In [ ]:
# Chart 2: Change% vs ATR (Risk-Reward Analysis)
import numpy as np
# Prepare a non-negative size column for markers (Plotly doesn't accept negative sizes)
size_col = None
if 'Remaining Move' in df_results.columns and not df_results['Remaining Move'].isna().all():
    # Use absolute values to ensure non-negative marker sizes (Remaining Move may be negative for reversal potential)
    size_vals = pd.to_numeric(df_results['Remaining Move'].abs(), errors='coerce')
    if size_vals.fillna(0).sum() == 0:
        size_col = None
    else:
        # Map sizes to a sensible pixel range for markers (10 to 40) to ensure visibility
        min_size, max_size = 10, 40
        filled = size_vals.fillna(0).values
        if filled.max() == filled.min():
            df_results['Remaining Move Size'] = np.where(np.isnan(size_vals), np.nan, (min_size + max_size) / 2)
        else:
            df_results['Remaining Move Size'] = np.interp(filled, (filled.min(), filled.max()), (min_size, max_size))
        size_col = 'Remaining Move Size'
# Build the figure and gracefully fallback to no 'size' if size values cause an error
try:
    fig2 = px.scatter(
        df_results.dropna(subset=['ATR', 'Change%']),
        x='ATR',
        y='Change%',
        color='Signal',
        size=size_col,
        size_max=40,
        hover_data=['Stock', 'Index', 'Change'],
        color_discrete_map={'Long': '#00cc96', 'Short': '#ef553b'},
        title='Risk-Reward Analysis: Change% vs ATR (size = Remaining Move potential)',
        labels={'ATR': 'Average True Range (Risk)', 'Change%': 'Change % (Return)'},
        height=600
)
except ValueError as e:
    # Some Plotly versions can still throw ValueError for invalid sizes. Fallback to ignoring size.
    if 'Invalid element(s) received for the \"size\" property' in str(e) or 'Invalid element(s) received for the \"size\" property' in str(e):
        print("Warning: invalid sizes in 'Remaining Move' — plotting without size markers")
        fig2 = px.scatter(
            df_results.dropna(subset=['ATR', 'Change%']),
            x='ATR',
            y='Change%',
            color='Signal',
            size=None,
            hover_data=['Stock', 'Index', 'Change'],
            color_discrete_map={'Long': '#00cc96', 'Short': '#ef553b'},
            title='Risk-Reward Analysis: Change% vs ATR (size = Remaining Move potential)',
            labels={'ATR': 'Average True Range (Risk)', 'Change%': 'Change % (Return)'},
            height=600
)
    else:
        raise
fig2.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Neutral")
fig2.update_layout(showlegend=True, hovermode='closest')
fig2.show()

In [ ]:
# Chart 3: Distribution by Index
index_summary = df_results.groupby(['Index', 'Signal']).size().unstack(fill_value=0)

fig3 = px.bar(
    index_summary.reset_index().melt(id_vars='Index', var_name='Signal', value_name='Count'),
    x='Index',
    y='Count',
    color='Signal',
    color_discrete_map={'Long': '#00cc96', 'Short': '#ef553b'},
    barmode='group',
    title='Opportunities by Index (Long vs Short)',
    labels={'Count': 'Number of Stocks', 'Index': 'Index'},
    height=500
)
fig3.update_layout(showlegend=True, hovermode='x unified')
fig3.show()

# Summary statistics
print("\n" + "=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
print(f"\nTotal Opportunities: {len(df_results)}")
print(f"  Long Signals:  {len(df_results[df_results['Signal'] == 'Long'])}")
print(f"  Short Signals: {len(df_results[df_results['Signal'] == 'Short'])}")
print(f"\nBy Index:")
print(index_summary)
print(f"\nAverage Change%: {df_results['Change%'].mean():.2f}%")
print(f"Average ATR: {df_results['ATR'].mean():.2f}")
print(f"Median Remaining Move: {df_results['Remaining Move'].median():.2f}")


In [ ]:
# Chart 4: Heatmap - Top stocks with key metrics
top_stocks = df_results.nlargest(20, 'Remaining Move').dropna(subset=['ATR', 'Remaining Move'])

heatmap_data = top_stocks[['Stock', 'Change%', 'ATR', 'Remaining Move']].set_index('Stock')
# Normalize for better heatmap visualization
heatmap_normalized = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

fig4 = go.Figure(data=go.Heatmap(
    z=heatmap_normalized.values,
    x=heatmap_normalized.columns,
    y=heatmap_normalized.index,
    colorscale='RdYlGn',
    text=heatmap_data.values.round(2),
    texttemplate='%{text}',
    textfont={"size": 10},
    colorbar=dict(title="Normalized Score")
))
fig4.update_layout(
    title='Top 20 Stocks - Key Metrics Heatmap (Normalized)',
    height=600,
    width=900
)
fig4.show()


In [ ]:
# Chart 5: Box plots - Distribution of key metrics by Signal Type
fig5 = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Change% Distribution', 'ATR Distribution', 'Remaining Move Distribution'),
    specs=[[{}, {}, {}]]
)

for signal in ['Long', 'Short']:
    data = df_results[df_results['Signal'] == signal]
    color = '#00cc96' if signal == 'Long' else '#ef553b'
    
    fig5.add_trace(
        go.Box(y=data['Change%'], name=f'{signal} Change%', marker=dict(color=color), boxmean='sd'),
        row=1, col=1
    )
    fig5.add_trace(
        go.Box(y=data['ATR'].dropna(), name=f'{signal} ATR', marker=dict(color=color), boxmean='sd'),
        row=1, col=2
    )
    fig5.add_trace(
        go.Box(y=data['Remaining Move'].dropna(), name=f'{signal} Rem Move', marker=dict(color=color), boxmean='sd'),
        row=1, col=3
    )

fig5.update_layout(height=500, showlegend=False, title_text="Distribution of Key Metrics")
fig5.update_yaxes(title_text="Change %", row=1, col=1)
fig5.update_yaxes(title_text="ATR", row=1, col=2)
fig5.update_yaxes(title_text="Remaining Move", row=1, col=3)
fig5.show()


## `%` Change probability (Stock Selection)
Gives the stock names which have the highest probability of providing you `x%` in intraday atleast on both long or short side based on the previous `N` days record

In [ ]:
# Gives you top 10 stocks from Nifty 500 which have the highest probability of giving atleast 0.99% on your investment based on the Long/Buy position. Data is based on past 60 trading days
# Test it for different time duration, index, % change and Long / Short criteria based on overall market trend
Intra.prob_by_percent_change(symbol = None, index = 500, time_period = 60, change_percent=1.0, sort_by='Long Probability', top_k = 10) 

## ATR 
If stock has crossed it's ATR on either side, it might reverse or if it hasn't, there is still a chance to move

In [ ]:
df = Intra.ATR_strategy(index = 'NIFTY 200', possible_reversal=False) # if possible_reversal is true, it'll just reverse the order of remaining move %
df.head(20)

# [Live Market]()

## Minutes LIVE Data 🤯🤯🤯🤯🤑🤑🤑🤑🥳️🥳️🥳️
Yup! Package now supports `[2,3,4,5,10,15,30,60]` minutes Live Data Too! Finally!!

In order to access the data, there is a unique code associated with every name and is known only to them. You need to write the code for the stock you want to access. This is one time process per stock. You can find that code as follows:

1. Login to `Kite` from your PC and open the chart of `5` minutes for any stock you want to analyse.
2. Click `Ctrl + Shift + I` in Chrome. A new small panel will get opened on the right hand side of screen.
3. Find **NETWORK** in the upper region of that panel. It'll be in the upper region only next to `Elements | Console | Sources | Network`. Click on **NETWORK**
4. Click `CTRL+R` to reload. You'll find something that starts with `5minute?user_id=` under `Name`. [You'll be looking at something like this. Highlighted `holdings` is equal to your `5minute?user_id=`](https://marketsetup.in/posts/zerodha-login/portfolio.png). Click on it and see `Headers` just like in the picture.
5. You'll see a Something like `Request URL: https://kite.zerodha.com/oms/instruments/historical/2513665/5minute?user_id=AB1234&oi=1&from=2022-01-09&to=2022-01-14`
6. The number after `/historical/2513665` is the one you are looking for. So in this case, it is `HAVELLS` code. Save it for future purpose.

### Please help me help you. I am trying to help more and more people. Not everyone can get this code easily and I can not get all the 1600 codes for everyone. If you are looking for a stock and find it's code, just send me a `feature request` or `raise an issue` and paste  the code along with scrip name there so that more and more people can be helped.

In [ ]:
# Kite = KiteZerodha(user_id = 'AB1234',password = "password@123",two_factor_pin = 123456) # You'll be logged out from your PC as much as I know
# df =  Kite.get_historical_minutes_data(name = 'HAVELLS', code = None, interval = 15, starting_from_date = None, no_days_back = 50, get_latest_only = False) # can also directly use code = 2513665
# # Get 15 minutes data of HINDPETRO starting from Today (including current candles) to the previous 50 days. If get_latest_only = True, it would give you 1 days day i.e Today
# df

## Market Sentiment: TICK, TRIN (Arm's Index) & 52W High-Low touching shares

In [ ]:
MS.get_live_sentiment()

## [Volatility Index: VIX](https://www.motilaloswal.com/blog-details/6-things-that-the-Volatility-Index-(VIX)-indicates-to-you../1929)

In [ ]:
NSE.get_VIX()

## MMI: [Market Mood Index](https://www.tickertape.in/market-mood-index)

Please click on the link above to get the values. All credits to [Ticker Tape Website](https://www.tickertape.in/)

In [ ]:
get_mmi(raw = True) # True means a simple Json shows current, last week, last month MMI

## Current Index Performance
Includes Nifty, Thematic and Sectoral indices and show `Top-N` sorted by change %

In [ ]:
NSE.current_indices_status(10) # Live as of market is open. works post market also

In [ ]:
NSE.open_nse_index('NIFTY ENERGY',) # Returns performance of stocks in a given index

In [ ]:
In.stock_current_index_performance('HAVELLS') # All the eligible indices (and their current performance) where this stock belongs to. Some stocks might not belong aywhere

## Stocks trading at their 52W Low / High

In [ ]:
NSE.stocks_at_52W(direction = 'low') # set direction = 'high' for stocks going up

# Futures and Options

## [Analyse Option Chain](https://www.quora.com/How-do-I-read-analyse-the-option-chain-of-a-stock-to-intraday-trade-with-clarity-NSE). Very helpful

In [ ]:
filtered_df = analyse_option_chain('TATACHEM', plot = True, fig_size=(21,6.7)) # Read the Doc String. Manually compare other parameters such as openInterest

# [Back Testing](https://www.investopedia.com/terms/b/backtesting.asp)
## Back Testing Stratagies to get the best Win %, Max Return Investment etc
Pass in the parameters dict for all. Check Docstring of each individual function for how a strategy works in buying or selling.

## CCI
`BT.backtest('cci', stocks=['DIVISLAB'])` , for individual stock data performance to tell how much a stock would have gained or lost

In [ ]:
cci_parameters = {'buying_thresh':-100, 'selling_thresh':100, 'window':20, 'cols':('OPEN','CLOSE','LOW','HIGH', 'DATE')} # check documentation of BT.cci()

# Using the CCI strategy, Return top gaining stocks which have traded atleast 365 recent days and are currently in nifty-500
BT.backtest('cci', min_days = 365, top_n=10, stocks = 'nifty_500', **cci_parameters)

## MA
Moving Average Strategy

In [ ]:
BT.backtest('ma',stocks='nifty_500',top_n=5)

## RSI

In [ ]:
BT.backtest('rsi',stocks='nifty_500', top_n = 5) 

## MACD

In [ ]:
BT.backtest('macd', top_n=5, stocks=['POWERINDIA','MINDACORP',"ATGL"]) # Individual stocks. If no sells have been made, it'll be empty

## Stochastic Oscillator
Faster signal than MACD. Pass in the paramerters to change results and modify to your ease

In [ ]:
BT.backtest('stochastic_osc',stocks='nifty_500', top_n = 15) 

# [Risk Management](https://www.youtube.com/watch?v=v3lfIdpKSiU)

You have a total budget of 40K, and a risk apetite of rs 400 per trade, You can set manual such as `entry = 3980.1` and `stop_loss=3780.5`. If intraday, set `leverage` parameter to `>1` 

In [ ]:
In.get_particulars('APOLLOHOSP',budget = 40000, risk = 400,)  #manually set parameters for entry, exit etc

In [ ]:
# ============================================================================
# NORMALIZE NUMPY TYPES TO NATIVE PYTHON TYPES FOR JSON-SAFE OUTPUT
# ============================================================================
# Problem: dicts with numpy scalars print like {'Buying Price': np.float64(7428.42)}
# Solution: Convert numpy types to native Python types using .item()
# ============================================================================

import numpy as np
import json

# Example trade dict with numpy scalars (as returned from calculations)
trade_config = {
    'Buying Price': np.float64(4052.55),
    'Stop-Loss %': np.float64(2.01),
    'Target %': np.float64(4.01),
    'Quantity': np.float64(4.0),
    'Stop-Loss Price': np.float64(3971.2248),
    'Trigger Price': np.float64(4215.2),
    'investment_required': np.float64(16210.194),
    'Risk Per Share': np.float64(81.32),
    'Profit Per Share': np.float64(162.65),
    'Max loss on this config': np.float64(325.29),
    'Max Gain on this config': np.float64(650.59),
    'Index': ['Nifty 100']
}

# ===== METHOD 1: Simple normalization for most dicts =====
def normalize_value(v):
    """Convert numpy/array types to native Python types recursively."""
    if isinstance(v, np.generic):  # numpy scalar
        return v.item()
    if isinstance(v, np.ndarray):  # numpy array
        return v.tolist()
    if isinstance(v, (list, tuple)):
        return [normalize_value(x) for x in v]
    if isinstance(v, dict):
        return {k: normalize_value(val) for k, val in v.items()}
    return v

# Apply normalization
trade_normalized = {k: normalize_value(v) for k, v in trade_config.items()}

print("=" * 80)
print("COMPARISON: Raw Dict vs Normalized Dict")
print("=" * 80)
print("\n1. BEFORE (with numpy types - shows np.float64(...)):")
print(f"   {trade_config}\n")

print("2. AFTER (normalized - clean Python types):")
print(f"   {trade_normalized}\n")

# ===== METHOD 2: Export to JSON (JSON-safe) =====
print("=" * 80)
print("EXPORT TO JSON (clean representation)")
print("=" * 80)
json_safe = json.dumps(trade_normalized, indent=2)
print(json_safe)

# ===== METHOD 3: Display as DataFrame with formatting (best for notebooks) =====
import pandas as pd

print("\n" + "=" * 80)
print("BEST FOR NOTEBOOKS: Formatted pandas DataFrame")
print("=" * 80)

df_trade = pd.DataFrame([trade_normalized])
# Reorder columns for readability
cols = [
    'Index', 'Buying Price', 'Stop-Loss %', 'Stop-Loss Price', 'Trigger Price',
    'Quantity', 'investment_required', 'Risk Per Share', 'Profit Per Share',
    'Max loss on this config', 'Max Gain on this config', 'Target %'
]
cols_present = [c for c in cols if c in df_trade.columns]
df_trade = df_trade[cols_present]

# Format display
style_map = {
    'Buying Price': '{:,.2f}',
    'Stop-Loss %': '{:.2f}%',
    'Target %': '{:.2f}%',
    'Stop-Loss Price': '{:,.2f}',
    'Trigger Price': '{:,.2f}',
    'investment_required': '{:,.2f}',
    'Risk Per Share': '{:,.2f}',
    'Profit Per Share': '{:,.2f}',
    'Max loss on this config': '{:,.2f}',
    'Max Gain on this config': '{:,.2f}'
}
display(df_trade.style.format(style_map).hide(axis='index'))

print("\n✓ Clean values - ready for export or further processing!")


# Connect to Online Brokers
First step towards Algo trading

## Zerodha / Kite

In [ ]:
Kite = KiteZerodha(user_id = 'AB1234',password = "password@123",two_factor_pin = 123456)
Kite.check_basic_info('holdings') # can check user info, positions, margins etc 

In [ ]:
# Create a logic to filter stocks based on time period of daily, week, month crossovers SMA20 and SMA50
# The idea is to find stocks which have crossed over in the last 'n' days only

DH = DataHandler()
# Corrected call: Removed the explicit 'DH' argument
data = DH.find_recent_golden_crossovers(days_back=30)
print(data)
